# 🌸 Marigold V2 360° Panorama Depth Estimation & 3D Reconstruction

GPU-accelerated, production-ready inference notebook for **[marigold_cli](https://github.com/A511-git/marigold_cli)**.

### 🌟 Pipeline Features
- **Pure Marigold V2 DiT Backbone**: Powered by Qwen-Image-Edit-2509 Diffusion Transformer + Huawei Bayer Lab single-step flow matching.
- **4-bit BitsAndBytes Quantization**: Fits comfortably within standard 12GB–16GB Kaggle GPUs (T4 / P100 / L4).
- **Automatic 2048x1024 Preprocessing**: Batch downscales all raw panoramas to standard equirectangular resolution (2048:1024) with high-quality area interpolation.
- **Output Folder Self-Contained RGB**: Automatically saves the downscaled `image.png` directly in each output scene directory.
- **Kaggle `/tmp` Model Caching**: Directs HuggingFace & Torch downloads to `/tmp` to avoid Kaggle working disk quota exhaustion.
- **12-Camera Overlap Alignment**: Solves pairwise closed-form linear least-squares scale & shift across overlapping tiles before Poisson blending.
- **Metric Camera Height Floor Calibration**: Calibrates affine log-depth against ground floor rays (default: 1.5m) to recover true physical meters.
- **Full Production Export**: Outputs `depth.npy` (Float32 in true meters), `depth.exr` (HDR), `depth_vis.png` (Colorized Turbo), `mask.png`, `normal_vis.png`, `image.png` (RGB), `pointcloud.ply` (Fast Binary Little-Endian 3D Point Cloud), and `splat.ply` (3D Gaussian Splatting).

In [ ]:
# =============================================================================
# 1. CONFIGURATION & PARAMETERS
# =============================================================================
import os
import torch

# Direct all downloads and model caches to Kaggle's fast /tmp directory
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/huggingface/transformers"
os.environ["HF_HUB_CACHE"] = "/tmp/huggingface/hub"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["MPLBACKEND"] = "Agg"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

# ---- Input & Output Paths ----
RAW_INPUT_DIR       = "/kaggle/input/datasets/newmailserver/panorama-imgs/panno"   # Raw dataset attached to notebook
PROCESSED_INPUT_DIR = "/kaggle/working/input_2048x1024"                           # Downscaled 2048x1024 images
WORK_DIR            = "/kaggle/working/marigold_cli"                             # Cloned marigold_cli repo
OUTPUT_DIR          = "/kaggle/working/output"                                   # Root output directory
UPLOAD_DIR          = "/kaggle/temp/upload"                                      # Staging dir containing output + metadata

# ---- Downscale Target Resolution ----
TARGET_WIDTH        = 2048
TARGET_HEIGHT       = 1024

# ---- Git Repository & Model Configuration ----
MARIGOLD_REPO       = "https://github.com/A511-git/marigold_cli.git"
MARIGOLD_CHECKPOINT = "huawei-bayerlab/marigold-v2-0"  # Local path or HuggingFace repo
# Options for BASE_MODEL:
# 1. "Meatfucker/Qwen-Image-Edit-bnb-nf4" : Pre-quantized 4-bit NF4 (~4.5 GB download, ultra-fast setup)
# 2. "Qwen/Qwen-Image-Edit-2509"          : Full official base model (quantizes dynamically)
BASE_MODEL          = "Meatfucker/Qwen-Image-Edit-bnb-nf4"
MODALITY            = "depth"                         # Options: depth, normals, albedo
QUANTIZATION        = "4bit"                          # Options: 4bit, 8bit, none

# ---- Inference & Metric Calibration Settings ----
USE_FP16            = True     # Use FP16/BF16 half precision for fast inference
SPLIT_RESOLUTION    = 512      # Resolution per perspective tile (512 or 1024)
BATCH_SIZE          = 1        # Batch size for perspective view inference
CAMERA_HEIGHT       = 1.5      # Camera mounting height above floor in meters (default 1.5m)
MIN_DEPTH           = 0.3      # Minimum physical depth clamp in meters
MAX_DEPTH           = 15.0     # Maximum physical depth clamp in meters
ALIGN_TILES         = True     # Pairwise least-squares scale & shift alignment across 12 tiles
SAVE_MAPS           = True     # Save depth_vis.png, depth.exr, mask.png, and image.png
SAVE_POINTS_PLY     = True     # Save 3D point cloud pointcloud.ply (Binary Little-Endian)
SAVE_DEBUG          = False    # Save 12 individual split perspective views and camera JSONs

# ---- Kaggle Dataset Upload Configuration (Optional) ----
KAGGLE_USERNAME     = "newmailserver"
DATASET_SLUG        = "marigold-pano-output"
UPLOAD_DIR          = "/kaggle/temp/upload"   # Staging dir containing output + metadata

# Create base directories
os.makedirs(PROCESSED_INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs("/tmp/huggingface", exist_ok=True)
os.makedirs("/tmp/torch", exist_ok=True)

print("✅ Config loaded successfully.")
print("Raw Input dir:        ", RAW_INPUT_DIR)
print("Processed Input dir:  ", PROCESSED_INPUT_DIR, f"({TARGET_WIDTH}x{TARGET_HEIGHT})")
print("Output dir:           ", OUTPUT_DIR)
print("Model Cache:          /tmp/huggingface, /tmp/torch")
print("Camera Height Metric: ", CAMERA_HEIGHT, "m")
print("Tile Alignment:       ", ALIGN_TILES)
print("Checkpoint:           ", MARIGOLD_CHECKPOINT)
print("Base Model:           ", BASE_MODEL)
print("Quantization:         ", QUANTIZATION)


In [ ]:
# =============================================================================
# 2. PREPROCESSING: DOWNSCALE PANORAMAS TO 2048x1024 (2048:1024)
# =============================================================================
import os
import glob
from pathlib import Path
import cv2
from tqdm import tqdm

print(f"Scanning raw images from: {RAW_INPUT_DIR}")
extensions = (
    os.path.join(RAW_INPUT_DIR, "**", "*.jpg"),
    os.path.join(RAW_INPUT_DIR, "**", "*.jpeg"),
    os.path.join(RAW_INPUT_DIR, "**", "*.png"),
    os.path.join(RAW_INPUT_DIR, "**", "*.webp"),
    os.path.join(RAW_INPUT_DIR, "**", "*.JPG"),
    os.path.join(RAW_INPUT_DIR, "**", "*.JPEG"),
    os.path.join(RAW_INPUT_DIR, "**", "*.PNG"),
    os.path.join(RAW_INPUT_DIR, "**", "*.WEBP")
)

image_files = []
for pattern in extensions:
    image_files.extend(glob.glob(pattern, recursive=True))

image_files = sorted(list(set(image_files)))
print(f"Found {len(image_files)} raw panorama image(s).")

processed_count = 0
for img_path in tqdm(image_files, desc="Downscaling to 2048x1024"):
    rel_path = os.path.relpath(img_path, RAW_INPUT_DIR)
    dest_path = os.path.join(PROCESSED_INPUT_DIR, rel_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    
    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Warning: Could not read image {img_path}")
        continue
    
    h, w = img.shape[:2]
    if (w, h) != (TARGET_WIDTH, TARGET_HEIGHT):
        # High-quality area interpolation for downscaling equirectangular images
        interp = cv2.INTER_AREA if (w > TARGET_WIDTH or h > TARGET_HEIGHT) else cv2.INTER_LANCZOS4
        img_resized = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=interp)
    else:
        img_resized = img
    
    cv2.imwrite(dest_path, img_resized)
    processed_count += 1

print(f"\n✅ Preprocessing complete: {processed_count} panorama(s) downscaled and saved to {PROCESSED_INPUT_DIR}.")


In [ ]:
# =============================================================================
# 3. INSTALLATION (UV SYNC FROM PYPROJECT.TOML)
# =============================================================================
import os

os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/huggingface/transformers"
os.environ["HF_HUB_CACHE"] = "/tmp/huggingface/hub"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["MPLBACKEND"] = "Agg"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

!pip install -q uv

%cd {WORK_DIR}/..
!rm -rf {WORK_DIR}
!git clone --depth 1 {MARIGOLD_REPO} {WORK_DIR}

%cd {WORK_DIR}
!uv sync

# Verify installation via uv virtual environment with /tmp cache
!HF_HOME=/tmp/huggingface TORCH_HOME=/tmp/torch uv run python -c "import torch, diffusers, cv2, standalone_marigold; print('All Marigold V2 modules and dependencies imported successfully via uv!')"

# Post-processing & KaggleHub dependencies in base environment
!pip install -q opencv-python Pillow numpy imageio kagglehub

print("\nInstall complete. HuggingFace & Torch cache configured to /tmp.")


In [ ]:
# =============================================================================
# 4. RUN MARIGOLD V2 PANORAMA INFERENCE
# =============================================================================
import os
import sys

%cd {WORK_DIR}

# Build CLI command flags
flags = []
if USE_FP16:
    flags.append("--fp16")
if SAVE_MAPS:
    flags.append("--maps")
if SAVE_POINTS_PLY:
    flags.append("--points_ply")
if SAVE_DEBUG:
    flags.append("--debug")
if ALIGN_TILES:
    flags.append("--align")
else:
    flags.append("--no-align")

flags.extend(["--camera_height", str(CAMERA_HEIGHT)])
flags.extend(["--min_depth", str(MIN_DEPTH)])
flags.extend(["--max_depth", str(MAX_DEPTH)])
flags.extend(["--split_resolution", str(SPLIT_RESOLUTION)])
flags.extend(["--batch_size", str(BATCH_SIZE)])

flag_str = " ".join(flags)

# Run single CLI execution on downscaled input dataset folder (loads model ONCE onto GPU)
!HF_HOME=/tmp/huggingface TRANSFORMERS_CACHE=/tmp/huggingface/transformers TORCH_HOME=/tmp/torch uv run python app.py \
    -i "{PROCESSED_INPUT_DIR}" \
    -o "{OUTPUT_DIR}" \
    -c "{MARIGOLD_CHECKPOINT}" \
    --base_model "{BASE_MODEL}" \
    -m "{MODALITY}" \
    -q "{QUANTIZATION}" \
    --device cuda \
    {flag_str}


In [ ]:
# =============================================================================
# 5. D2P POST-PROCESSING (3D POINT CLOUDS & SURFACE NORMALS)
# =============================================================================
import os
import glob
from typing import Optional, Tuple, Union
import numpy as np
from PIL import Image
import cv2

def image_uv(width: int, height: int, left=None, top=None, right=None, bottom=None, dtype=np.float32) -> np.ndarray:
    if left is None: left = 0
    if top is None: top = 0
    if right is None: right = width
    if bottom is None: bottom = height
    u = np.linspace((left + 0.5) / width, (right - 0.5) / width, right - left, dtype=dtype)
    v = np.linspace((top + 0.5) / height, (bottom - 0.5) / height, bottom - top, dtype=dtype)
    u, v = np.meshgrid(u, v, indexing='xy')
    return np.stack([u, v], axis=2)

def sphere_uv2dirs(uv: np.ndarray) -> np.ndarray:
    theta = (1.0 - uv[..., 0]) * (2.0 * np.pi)
    phi = uv[..., 1] * np.pi
    directions = np.stack([
        np.sin(phi) * np.cos(theta),
        np.sin(phi) * np.sin(theta),
        np.cos(phi)
    ], axis=-1)
    return directions

def points_to_normals(point: np.ndarray, mask: Optional[np.ndarray] = None) -> Union[np.ndarray, Tuple[np.ndarray, np.ndarray]]:
    height, width = point.shape[-3:-1]
    has_mask = mask is not None
    if mask is None:
        mask = np.ones((height, width), dtype=bool)
    else:
        mask = mask.astype(bool)
    mask_pad = np.zeros((height + 2, width + 2), dtype=bool)
    mask_pad[1:-1, 1:-1] = mask
    pts = np.zeros((height + 2, width + 2, 3), dtype=point.dtype)
    pts[1:-1, 1:-1, :] = point
    up = pts[:-2, 1:-1, :] - pts[1:-1, 1:-1, :]
    down = pts[2:, 1:-1, :] - pts[1:-1, 1:-1, :]
    left = pts[1:-1, :-2, :] - pts[1:-1, 1:-1, :]
    right = pts[1:-1, 2:, :] - pts[1:-1, 1:-1, :]
    mask_up = mask_pad[:-2, 1:-1]
    mask_down = mask_pad[2:, 1:-1]
    mask_left = mask_pad[1:-1, :-2]
    mask_right = mask_pad[1:-1, 2:]
    cross1 = np.cross(left, up)
    cross2 = np.cross(up, right)
    cross3 = np.cross(right, down)
    cross4 = np.cross(down, left)
    normals = (
        cross1 * (mask_left & mask_up)[..., None] +
        cross2 * (mask_up & mask_right)[..., None] +
        cross3 * (mask_right & mask_down)[..., None] +
        cross4 * (mask_down & mask_left)[..., None]
    )
    normals = normals / np.linalg.norm(normals, axis=-1, keepdims=True).clip(min=1e-8)
    normals_mask = mask & ((mask_left & mask_up) | (mask_up & mask_right) | (mask_right & mask_down) | (mask_down & mask_left))
    if has_mask:
        return normals, normals_mask
    return normals

def depth2points(depth: np.ndarray, mask: Optional[np.ndarray] = None) -> Union[np.ndarray, Tuple[np.ndarray, np.ndarray]]:
    height, width = depth.shape[-2:]
    uv = image_uv(width=width, height=height, dtype=depth.dtype)
    directions = sphere_uv2dirs(uv)
    points = directions * depth[..., None]
    if mask is not None:
        return points, mask
    return points

def colorize_normals(normals: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
    norm_rgb = ((normals + 1.0) * 0.5 * 255.0).clip(0, 255).astype(np.uint8)
    if mask is not None:
        norm_rgb = np.where(mask[..., None], norm_rgb, 0)
    return norm_rgb

# Post-process all output subfolders
out_subdirs = sorted([d for d in glob.glob(os.path.join(OUTPUT_DIR, "**"), recursive=True) if os.path.isfile(os.path.join(d, "depth.npy"))])
print(f"Found {len(out_subdirs)} completed scenes for D2P post-processing.")

for folder in out_subdirs:
    d_npy = os.path.join(folder, "depth.npy")
    if not os.path.isfile(d_npy): continue
    depth = np.load(d_npy)
    m_path = os.path.join(folder, "mask.png")
    mask = (cv2.imread(m_path, cv2.IMREAD_GRAYSCALE) > 128) if os.path.isfile(m_path) else None
    pts, _ = depth2points(depth, mask=mask)
    np.save(os.path.join(folder, "points.npy"), pts)
    norms, nmask = points_to_normals(pts, mask=mask)
    norm_vis = colorize_normals(norms, mask=nmask)
    cv2.imwrite(os.path.join(folder, "normal_vis.png"), cv2.cvtColor(norm_vis, cv2.COLOR_RGB2BGR))
    rel = os.path.relpath(folder, OUTPUT_DIR)
    print(f"  ✅ Post-processed surface normals & point maps: {rel}")

print("✨ All D2P 3D reconstruction outputs generated!")


In [ ]:
# =============================================================================
# 6. 3D GAUSSIAN SPLAT GENERATOR (splat.ply)
# =============================================================================
import os
import glob
import numpy as np
import cv2

def save_gaussian_splat_ply(
    filepath: str,
    points: np.ndarray,
    colors: np.ndarray,
    scales: np.ndarray,
    quats: np.ndarray,
    opacities: np.ndarray = None
):
    """
    Exports points as standard 3D Gaussian Splatting binary PLY format.
    Compatible with SuperSplat, PlayCanvas, Luma AI, and WebGL 3DGS Viewers.
    """
    N = len(points)
    if N == 0:
        print(f"⚠️ No points to save for {filepath}")
        return

    if opacities is None:
        opacities = np.full((N, 1), 4.5, dtype=np.float32)  # High opacity logit (~0.989)

    # Spherical Harmonics DC (Degree 0) from RGB: f_dc = (rgb/255 - 0.5) / 0.28209479177387814
    sh_dc = ((colors.astype(np.float32) / 255.0) - 0.5) / 0.28209479177387814
    normals = np.zeros((N, 3), dtype=np.float32)

    dtype = [
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
        ('nx', 'f4'), ('ny', 'f4'), ('nz', 'f4'),
        ('f_dc_0', 'f4'), ('f_dc_1', 'f4'), ('f_dc_2', 'f4'),
        ('opacity', 'f4'),
        ('scale_0', 'f4'), ('scale_1', 'f4'), ('scale_2', 'f4'),
        ('rot_0', 'f4'), ('rot_1', 'f4'), ('rot_2', 'f4'), ('rot_3', 'f4'),
    ]

    elements = np.empty(N, dtype=dtype)
    elements['x'] = points[:, 0]
    elements['y'] = points[:, 1]
    elements['z'] = points[:, 2]
    elements['nx'] = normals[:, 0]
    elements['ny'] = normals[:, 1]
    elements['nz'] = normals[:, 2]
    elements['f_dc_0'] = sh_dc[:, 0]
    elements['f_dc_1'] = sh_dc[:, 1]
    elements['f_dc_2'] = sh_dc[:, 2]
    elements['opacity'] = opacities[:, 0]
    elements['scale_0'] = scales[:, 0]
    elements['scale_1'] = scales[:, 1]
    elements['scale_2'] = scales[:, 2]
    elements['rot_0'] = quats[:, 0]
    elements['rot_1'] = quats[:, 1]
    elements['rot_2'] = quats[:, 2]
    elements['rot_3'] = quats[:, 3]

    header = f"""ply
format binary_little_endian 1.0
element vertex {N}
property float x
property float y
property float z
property float nx
property float ny
property float nz
property float f_dc_0
property float f_dc_1
property float f_dc_2
property float opacity
property float scale_0
property float scale_1
property float scale_2
property float rot_0
property float rot_1
property float rot_2
property float rot_3
end_header
"""
    with open(filepath, 'wb') as f:
        f.write(header.encode('ascii'))
        f.write(elements.tobytes())

def depth_to_spherical_gaussians(depth: np.ndarray, rgb: np.ndarray, mask: np.ndarray = None, stride: int = 1, global_scale: float = 1.2, disc_thickness: float = 0.2):
    H, W = depth.shape[:2]
    if rgb.shape[:2] != (H, W):
        rgb = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_AREA)

    if stride > 1:
        depth = depth[::stride, ::stride]
        rgb = rgb[::stride, ::stride]
        if mask is not None:
            mask = mask[::stride, ::stride]
        H, W = depth.shape[:2]

    u = np.linspace(0.5 / W, 1.0 - 0.5 / W, W, dtype=np.float32)
    v = np.linspace(0.5 / H, 1.0 - 0.5 / H, H, dtype=np.float32)
    u_grid, v_grid = np.meshgrid(u, v)

    theta = (1.0 - u_grid) * (2.0 * np.pi)
    phi = v_grid * np.pi

    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)

    dx = sin_phi * cos_theta
    dy = sin_phi * sin_theta
    dz = cos_phi

    dirs = np.stack([dx, dy, dz], axis=-1)
    pts = (dirs * depth[..., None]).astype(np.float32)

    t1 = np.stack([-sin_theta, cos_theta, np.zeros_like(theta)], axis=-1)
    t2 = np.stack([cos_phi * cos_theta, cos_phi * sin_theta, -sin_phi], axis=-1)

    d_theta = (2.0 * np.pi) / W
    d_phi = np.pi / H

    s1 = depth * (d_theta * sin_phi.clip(1e-3)) * global_scale
    s2 = depth * d_phi * global_scale
    s3 = disc_thickness * np.minimum(s1, s2)

    log_scales = np.log(np.stack([s1, s2, s3], axis=-1).clip(1e-5, 1e2)).astype(np.float32)

    R00, R01, R02 = t1[..., 0], t2[..., 0], dx
    R10, R11, R12 = t1[..., 1], t2[..., 1], dy
    R20, R21, R22 = t1[..., 2], t2[..., 2], dz

    tr = R00 + R11 + R22
    qw = np.sqrt(np.maximum(0.0, 1.0 + tr)) / 2.0
    qx = (R21 - R12) / (4.0 * np.maximum(qw, 1e-6))
    qy = (R02 - R20) / (4.0 * np.maximum(qw, 1e-6))
    qz = (R10 - R01) / (4.0 * np.maximum(qw, 1e-6))

    quats = np.stack([qw, qx, qy, qz], axis=-1).astype(np.float32)
    q_norm = np.linalg.norm(quats, axis=-1, keepdims=True).clip(1e-6)
    quats = quats / q_norm

    valid = np.isfinite(depth) & (depth > 0.01) & (depth < 1000.0)
    if mask is not None:
        valid = valid & mask

    flat_pts = pts[valid].reshape(-1, 3)
    flat_rgb = rgb[valid].reshape(-1, 3)
    flat_scales = log_scales[valid].reshape(-1, 3)
    flat_quats = quats[valid].reshape(-1, 4)

    return flat_pts, flat_rgb, flat_scales, flat_quats

# Generate splat.ply for all output scenes
out_subdirs = sorted([d for d in glob.glob(os.path.join(OUTPUT_DIR, "**"), recursive=True) if os.path.isfile(os.path.join(d, "depth.npy"))])
print(f"Found {len(out_subdirs)} completed scenes for 3D Gaussian Splat export.")

for folder in out_subdirs:
    d_npy = os.path.join(folder, "depth.npy")
    if not os.path.isfile(d_npy): continue
    depth = np.load(d_npy)
    m_path = os.path.join(folder, "mask.png")
    mask = (cv2.imread(m_path, cv2.IMREAD_GRAYSCALE) > 128) if os.path.isfile(m_path) else None
    
    # Read downscaled image saved in output directory
    img_path = os.path.join(folder, "image.png")
    rgb_img = None
    if os.path.isfile(img_path):
        rgb_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    else:
        rel = os.path.relpath(folder, OUTPUT_DIR)
        for ext in (".jpg", ".png", ".jpeg", ".webp", ".JPG", ".PNG", ".JPEG", ".WEBP"):
            processed_p = os.path.join(PROCESSED_INPUT_DIR, rel + ext)
            if os.path.isfile(processed_p):
                rgb_img = cv2.cvtColor(cv2.imread(processed_p), cv2.COLOR_BGR2RGB)
                break
    
    if rgb_img is None:
        rgb_img = np.full((depth.shape[0], depth.shape[1], 3), 200, dtype=np.uint8)

    pts, cols, scs, qts = depth_to_spherical_gaussians(depth, rgb_img, mask=mask, stride=1, global_scale=1.2)
    out_splat_path = os.path.join(folder, "splat.ply")
    save_gaussian_splat_ply(out_splat_path, pts, cols, scs, qts)
    print(f"  ✅ Generated 3D Gaussian Splat: {os.path.basename(folder)}/splat.ply ({len(pts):,} splats)")

print("🎉 All 3D Gaussian Splats (splat.ply) generated successfully!")

In [ ]:
# =============================================================================
# 7. KAGGLEHUB DATASET EXPORT & UPLOAD
# =============================================================================
import json
import shutil
import os
import kagglehub

ENABLE_UPLOAD = False   # Set to True when ready to export to Kaggle Datasets

if ENABLE_UPLOAD:
    print(f"Staging results from {OUTPUT_DIR} to {UPLOAD_DIR}...")
    if os.path.exists(UPLOAD_DIR):
        shutil.rmtree(UPLOAD_DIR)
    shutil.copytree(OUTPUT_DIR, UPLOAD_DIR)

    dataset_meta = {
        "title": "Marigold V2 360 Panorama Depth Output",
        "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(dataset_meta, f, indent=2)

    print(f"Uploading dataset to {KAGGLE_USERNAME}/{DATASET_SLUG} via KaggleHub...")
    kagglehub.dataset_upload(
        handle=f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
        local_dataset_dir=UPLOAD_DIR,
        version_notes="Marigold 360 Depth & Point Cloud Generation"
    )
    print("🎉 Kaggle dataset uploaded successfully!")
else:
    print("Dataset upload skipped (ENABLE_UPLOAD = False).")
